# Lab 11a · What is this model allowed to do?

[Open in Colab](https://colab.research.google.com/github/AIGuruOne/OQ-labs/blob/main/notebooks/11a_mcp_basics.ipynb)

**Day 4 · S22 · 20 min** · Runs on a laptop or Colab. **No GPU, no model call, no API key.**

## What you will be able to do after this

**Tell what a model is allowed to do by looking at how its tool server was started —
not at the model, the prompt or the client — and show that the same server file offers
different tools depending on that one setting.**

**You will know it landed** when you can answer both of these:

> *"Where is the write actually stopped?"* and
> *"What could a cleverer prompt do about it?"*

**Where you use it:**
S26 — you build this server.
S29 governance pack, template 4 — approval points.
S24 — your blast-radius sheet: the row for each tool is decided here.

---

Two servers are running in a moment. **Same file, same machine, started two different
ways.** Everything below is about what a client can and cannot get out of them.

**A correct result looks like:** the read-only server offers three tools and the other
four; the write is refused outright by the first; the second stops and asks a person;
and the audit log ends with three lines telling the whole story.

> All data in this lab is synthetic. No real OQ material anywhere.

## 1. Start both servers

The mock ERP, then the MCP server twice — once read-only, once with writes enabled.

In [ ]:
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mcp==2.2.0"], check=True)
    subprocess.run(["git", "clone", "-q", "https://github.com/AIGuruOne/OQ-labs.git"], check=False)
    sys.path.insert(0, "OQ-labs/notebooks")

import mcp_lab as mcp
mcp.start()

## 2. What does each one offer a client?

This is the whole lab in one comparison. Look at the last line of each list.

In [ ]:
print(mcp.tools("read_only"))
print()
print(mcp.tools("writes"))

The fourth tool **does not exist** as far as a client of the first server is concerned.
Not hidden, not permission-denied — absent from the list it is given.

Same file on disk. The only difference is one flag at start-up.

## 3. A read tool, for comparison

Both servers answer this identically. Nothing controversial happens here.

In [ ]:
print(mcp.call("get_equipment", server="read_only", tag="P-1201A")[:260])

## 4. Now ask the read-only server to write

We send a complete, valid, well-formed work order request. Everything about it is correct.

In [ ]:
print(mcp.call("raise_work_order", server="read_only", **mcp.WORK_ORDER))

Read that refusal again, because it answers the second question in the header:

> *"A person restarts it with `--enable-writes`; **nothing a client sends can**."*

There is no prompt, no argument, no clever framing that gets past this. The check
happens in the server, before the tool runs, based on how a **person** started it.
That is what makes it a control rather than a preference.

## 5. The same request, to the server that allows writes

In [ ]:
print(mcp.request_write())

It still did not write.

It stopped and sent back a form for a person — and notice what is on that form: the
**exact** request that will be sent, what the server checked about the equipment first,
and the warning that the work order cannot be cancelled afterwards. A person is being
asked to approve something specific, not to click yes on something vague.

## 6. A person answers

In [ ]:
print(mcp.answer("approve", approver="planner:salim", reason="seal history checked"))

## 7. The audit log

Three calls, three lines. This is what an auditor reads.

In [ ]:
print(mcp.audit(3))

Each line says what was asked, what class of access it needed, what happened, and
whether a person approved it. The refused attempts are logged **just as loudly** as
the successful one — a control that leaves no trace of what it stopped is not
evidence of anything.

## 8. What you take away

**The tool list is the security boundary.** Not the prompt, not the model, not the
client. If a tool is not in the list a server hands out, no amount of prompting reaches
it — and if it *is* in the list, a prompt eventually will.

**Two questions answer themselves now.** Where is the write stopped? In the server,
before the tool runs. What could a cleverer prompt do? Nothing.

**"Nothing writes without a person" is a thing you can point at.** A refusal, a form
with the exact payload on it, an approver's name, and a log line. Not a policy
sentence — four artifacts.

### Take this to the rest of the week

| Where | What it decides |
|---|---|
| **S24** blast-radius sheet | one row per tool: read or write, and who approves it |
| **S26** | you build this server, and this is the table you are implementing |
| **S29** governance pack, template 4 | approval points, with the audit line as the evidence |

In [ ]:
mcp.stop()
print("11a done - the tool list is the boundary, and a person is in the write.")